In [ ]:
import matplotlib.pyplot as plt

# Data
periods = ["2024 avg", "2025 avg", "2026/Q1", "2026/Q2"]
values = [8.71, 7.24, 7.96, 8.44]

colors = ["#5e35b1", "#1565c0", "#ef6c00", "#d81b60"]
benchmark = 7.00

# Create chart
fig, ax = plt.subplots(figsize=(8, 5))

bars = ax.bar(
    periods,
    values,
    color=colors,
    width=0.5,
    zorder=3
)

# Value labels
for bar, value in zip(bars, values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        value + 0.08,
        f"{value:.2f}%",
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold"
    )

# Benchmark line
ax.axhline(
    benchmark,
    color="#c62828",
    linestyle="--",
    linewidth=2,
    label=f"MYTO benchmark ({benchmark:.2f}%)",
    zorder=4
)

# Styling
ax.set_ylabel("Transmission Loss Factor (%)", fontsize=12)
ax.set_ylim(0, 10)
ax.set_yticks(range(0, 11, 2))

ax.tick_params(axis="both", labelsize=11)
ax.grid(axis="y", linestyle=":", color="#cccccc", zorder=0)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(loc="upper right", frameon=False, fontsize=10)

plt.tight_layout()
plt.savefig("tcn_transmission_loss.png", dpi=300, bbox_inches="tight")
plt.show()

"""
Technical-loss simulation of a representative Nigerian 11 kV radial feeder.

Two scenarios:
1. Baseline: 50 mm² AAC, load PF = 0.75, back-loaded demand.
2. Intervention: 150 mm² AAC, load PF = 0.75, a midpoint
   capacitor bank sized approximately to target PF = 0.95, and
   evenly distributed demand.

IMPORTANT:
- This is an illustrative 11 kV feeder model, not a calibrated utility feeder.
- The 33/11 kV substation transformer is represented by an ideal
  11 kV external-grid/slack bus. Transformer losses are excluded.
- Total active load is 2.0 MW, not 2.0 MVA.
- The capacitor is sized analytically; source PF is checked after the
  power-flow solution.
"""

from pathlib import Path
import json
import numpy as np
import pandapower as pp


# ============================================================
# 1. MODEL PARAMETERS
# ============================================================

N_DT = 10
SPAN_KM = 1.5
TOTAL_LOAD_MW = 2.0
V_KV = 11.0
F_HZ = 50.0

BASELINE_PF = 0.75
TARGET_PF = 0.95

# Representative conductor parameters, ohm/km and kA.
# Verify against the actual conductor manufacturer's data
# before making engineering design recommendations.
COND = {
    "aged_50mm2": {
        "r_ohm_per_km": 0.6740,
        "x_ohm_per_km": 0.372,
        "max_i_ka": 0.180,
    },
    "upgraded_150mm2": {
        "r_ohm_per_km": 0.2210,
        "x_ohm_per_km": 0.335,
        "max_i_ka": 0.375,
    },
}

# Illustrative economic assumptions, not verified tariff data.
TARIFF_NAIRA_PER_KWH = 209.5
UTILISATION_FACTOR = 0.60
HOURS_PER_YEAR = 8760


# ============================================================
# 2. LOAD PROFILES
# ============================================================

weights_backloaded = np.array([
    0.05, 0.06, 0.07, 0.08, 0.09,
    0.10, 0.12, 0.14, 0.15, 0.14
], dtype=float)

weights_backloaded /= weights_backloaded.sum()

load_baseline = (
    TOTAL_LOAD_MW * weights_backloaded
).tolist()

load_balanced = [
    TOTAL_LOAD_MW / N_DT
] * N_DT


# ============================================================
# 3. CAPACITOR SIZING
# ============================================================

# Approximate compensation needed to improve the load PF
# from 0.75 to 0.95 at 2 MW.
#
# Qc = P * [tan(acos(PF_initial)) - tan(acos(PF_target))]
#
# This calculation ignores feeder losses and voltage variation.
# The actual source PF must therefore be checked after load flow.

capacitor_mvar = TOTAL_LOAD_MW * (
    np.tan(np.arccos(BASELINE_PF))
    - np.tan(np.arccos(TARGET_PF))
)

print(f"Calculated capacitor rating: {capacitor_mvar:.4f} MVAr")


# ============================================================
# 4. NETWORK CONSTRUCTION
# ============================================================

def build_network(
    conductor_key,
    load_pf,
    load_profile,
    capacitor_mvar=0.0,
):
    if conductor_key not in COND:
        raise ValueError(f"Unknown conductor: {conductor_key}")

    if not 0 < load_pf <= 1:
        raise ValueError("Power factor must be in (0, 1].")

    if len(load_profile) != N_DT:
        raise ValueError(f"Expected {N_DT} load values.")

    if any(p < 0 for p in load_profile):
        raise ValueError("Load values must be non-negative.")

    if capacitor_mvar < 0:
        raise ValueError("Capacitor rating must be non-negative.")

    net = pp.create_empty_network(f_hz=F_HZ)

    # The slack bus represents the 11 kV bus at the injection
    # substation. The 33/11 kV transformer is not explicitly modelled.
    bus0 = pp.create_bus(
        net,
        vn_kv=V_KV,
        name="Injection Substation 11 kV Bus",
    )

    pp.create_ext_grid(
        net,
        bus=bus0,
        vm_pu=1.0,
        name="Grid Source",
    )

    buses = [bus0]
    conductor = COND[conductor_key]

    for i in range(1, N_DT + 1):

        distance_km = i * SPAN_KM

        bus = pp.create_bus(
            net,
            vn_kv=V_KV,
            name=f"DT-{i:02d} at {distance_km:.1f} km",
        )

        buses.append(bus)

        pp.create_line_from_parameters(
            net,
            from_bus=buses[i - 1],
            to_bus=bus,
            length_km=SPAN_KM,
            r_ohm_per_km=conductor["r_ohm_per_km"],
            x_ohm_per_km=conductor["x_ohm_per_km"],
            c_nf_per_km=10.0,
            max_i_ka=conductor["max_i_ka"],
            name=f"Feeder Span {i}",
        )

        p_mw = float(load_profile[i - 1])

        # Lagging reactive load for the specified load PF.
        q_mvar = p_mw * np.tan(np.arccos(load_pf))

        pp.create_load(
            net,
            bus=bus,
            p_mw=p_mw,
            q_mvar=q_mvar,
            name=f"DT Load {i:02d}",
        )

    if capacitor_mvar > 0:

        # Midpoint: bus 5, located 7.5 km from the source.
        mid_bus = buses[N_DT // 2]

        # In pandapower, negative q_mvar represents capacitive
        # reactive-power injection for a shunt.
        pp.create_shunt(
            net,
            bus=mid_bus,
            p_mw=0.0,
            q_mvar=-capacitor_mvar,
            name="Midpoint Capacitor Bank",
        )

    return net


# ============================================================
# 5. RUN AND EVALUATE A SCENARIO
# ============================================================

def run_scenario(
    name,
    conductor_key,
    load_pf,
    load_profile,
    capacitor_mvar=0.0,
):
    net = build_network(
        conductor_key=conductor_key,
        load_pf=load_pf,
        load_profile=load_profile,
        capacitor_mvar=capacitor_mvar,
    )

    pp.runpp(
        net,
        algorithm="nr",
        calculate_voltage_angles=True,
    )

    if not net.converged:
        raise RuntimeError(f"Power flow did not converge: {name}")

    total_load_mw = float(net.load["p_mw"].sum())

    # Use actual power-flow results, not rounded values.
    line_losses_mw = float(net.res_line["pl_mw"].sum())

    # Active power supplied by the external grid.
    source_p_mw = float(net.res_ext_grid["p_mw"].sum())
    source_q_mvar = float(net.res_ext_grid["q_mvar"].sum())

    source_s_mva = np.hypot(source_p_mw, source_q_mvar)

    source_pf = (
        abs(source_p_mw) / source_s_mva
        if source_s_mva > 0
        else 1.0
    )

    # Positive source Q indicates lagging demand;
    # negative source Q indicates leading operation.
    if abs(source_q_mvar) < 1e-8:
        source_pf_type = "approximately unity"
    elif source_q_mvar > 0:
        source_pf_type = "lagging"
    else:
        source_pf_type = "leading"

    # Exclude the source bus when assessing customer-side voltages.
    feeder_voltages = net.res_bus["vm_pu"].iloc[1:]

    voltage_min = float(feeder_voltages.min())
    voltage_max = float(feeder_voltages.max())

    # Loading as a percentage of each line's configured current limit.
    line_loading = net.res_line["loading_percent"].to_numpy()

    result = {
        "scenario": name,
        "conductor": conductor_key,
        "load_power_factor": load_pf,
        "capacitor_mvar": capacitor_mvar,
        "total_load_mw": total_load_mw,
        "line_losses_mw": line_losses_mw,
        "source_active_power_mw": source_p_mw,
        "source_reactive_power_mvar": source_q_mvar,
        "source_power_factor": source_pf,
        "source_power_factor_type": source_pf_type,
        "technical_loss_pct": (
            line_losses_mw / source_p_mw * 100
            if source_p_mw > 0 else 0.0
        ),
        "min_voltage_pu": voltage_min,
        "max_voltage_pu": voltage_max,
        "bus_voltages_pu": [
            float(v) for v in net.res_bus["vm_pu"]
        ],
        "line_loading_pct": [
            float(v) for v in line_loading
        ],
        "max_line_loading_pct": float(line_loading.max()),
        "overloaded_line_count": int(
            np.sum(line_loading > 100.0)
        ),
    }

    return result, net


# ============================================================
# 6. SCENARIO COMPARISON
# ============================================================

baseline, net_baseline = run_scenario(
    name="Baseline",
    conductor_key="aged_50mm2",
    load_pf=BASELINE_PF,
    load_profile=load_baseline,
    capacitor_mvar=0.0,
)

intervention, net_intervention = run_scenario(
    name="Intervention",
    conductor_key="upgraded_150mm2",
    load_pf=BASELINE_PF,
    load_profile=load_balanced,
    capacitor_mvar=capacitor_mvar,
)


# ============================================================
# 7. TECHNICAL-LOSS REDUCTION
# ============================================================

loss_reduction_mw = (
    baseline["line_losses_mw"]
    - intervention["line_losses_mw"]
)

loss_reduction_percentage_points = (
    baseline["technical_loss_pct"]
    - intervention["technical_loss_pct"]
)

loss_reduction_relative_pct = (
    loss_reduction_mw / baseline["line_losses_mw"] * 100
    if baseline["line_losses_mw"] > 0
    else 0.0
)


# ============================================================
# 8. ILLUSTRATIVE ANNUALISED ENERGY AND VALUE
# ============================================================

# This assumes the peak-load loss difference scales linearly
# with average feeder utilisation. Actual losses generally scale
# nonlinearly with current, so this is a screening estimate.

average_loss_reduction_mw = (
    loss_reduction_mw * UTILISATION_FACTOR
)

annual_energy_saved_mwh = (
    average_loss_reduction_mw * HOURS_PER_YEAR
)

annual_naira_saved = (
    annual_energy_saved_mwh
    * 1000
    * TARIFF_NAIRA_PER_KWH
)


# ============================================================
# 9. OUTPUT AND FILE EXPORT
# ============================================================

results = {
    "baseline": baseline,
    "intervention": intervention,
    "comparison": {
        "peak_loss_reduction_mw": loss_reduction_mw,
        "loss_reduction_percentage_points":
            loss_reduction_percentage_points,
        "loss_reduction_relative_pct":
            loss_reduction_relative_pct,
        "annual_energy_saved_mwh_estimate":
            annual_energy_saved_mwh,
        "annual_naira_saved_million_estimate":
            annual_naira_saved / 1e6,
    },
    "assumptions": {
        "total_active_load_mw": TOTAL_LOAD_MW,
        "number_of_dt_load_points": N_DT,
        "feeder_length_km": N_DT * SPAN_KM,
        "baseline_load_pf": BASELINE_PF,
        "target_load_pf_for_capacitor_sizing": TARGET_PF,
        "capacitor_mvar": capacitor_mvar,
        "tariff_naira_per_kwh": TARIFF_NAIRA_PER_KWH,
        "utilisation_factor": UTILISATION_FACTOR,
        "hours_per_year": HOURS_PER_YEAR,
        "transformer_modelled": False,
        "load_flow_converged": True,
    },
}

output_dir = Path("data")
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "technical_loss_results.json"

with output_file.open("w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, allow_nan=False)

print("\n========== TECHNICAL LOSS RESULTS ==========")

for scenario in [baseline, intervention]:
    print(f"\n{scenario['scenario']}")
    print("-" * 40)
    print(f"Total load:          {scenario['total_load_mw']:.4f} MW")
    print(f"Line losses:         {scenario['line_losses_mw']:.6f} MW")
    print(f"Technical losses:    {scenario['technical_loss_pct']:.3f}%")
    print(f"Source PF:           {scenario['source_power_factor']:.4f}")
    print(f"PF type:             {scenario['source_power_factor_type']}")
    print(f"Minimum voltage:     {scenario['min_voltage_pu']:.4f} pu")
    print(f"Maximum voltage:     {scenario['max_voltage_pu']:.4f} pu")
    print(f"Maximum line loading:{scenario['max_line_loading_pct']:.2f}%")
    print(f"Overloaded spans:    {scenario['overloaded_line_count']}")

print("\n========== COMPARISON ==========")
print(
    "Loss reduction: "
    f"{loss_reduction_mw * 1000:.3f} kW"
)
print(
    "Loss reduction: "
    f"{loss_reduction_percentage_points:.3f} percentage points"
)
print(
    "Relative reduction in peak losses: "
    f"{loss_reduction_relative_pct:.2f}%"
)
print(
    "Estimated annual energy saved: "
    f"{annual_energy_saved_mwh:.2f} MWh"
)
print(
    "Estimated annual value: "
    f"NGN {annual_naira_saved / 1e6:.2f} million"
)
print(f"\nResults saved to: {output_file.resolve()}")
print("\nFull JSON results:")
print(json.dumps(results, indent=2))



import sys
import subprocess

subprocess.run([
    sys.executable, "-m", "ensurepip", "--upgrade"
], check=True)

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ============================================================
# 1. INPUT DATA
# NOTE: Replace illustrative values with actual pandapower
# power-flow results before reporting as simulated findings.
# ============================================================

distance_km = np.arange(0, 15.1, 1.5)

baseline = np.array([
    1.000, 0.971, 0.943, 0.916, 0.892, 0.869,
    0.850, 0.833, 0.820, 0.811, 0.807
])

intervention = np.array([
    1.000, 0.995, 0.991, 0.988, 0.986, 0.984,
    0.980, 0.977, 0.974, 0.973, 0.972
])

voltage_min = 0.95

# ============================================================
# 2. DATA VALIDATION
# ============================================================

if not (
    len(distance_km) == len(baseline) == len(intervention)
):
    raise ValueError(
        "Distance and voltage arrays must have equal lengths."
    )

if not (
    np.all(np.isfinite(distance_km))
    and np.all(np.isfinite(baseline))
    and np.all(np.isfinite(intervention))
):
    raise ValueError("Input arrays contain NaN or infinity.")

if np.any(np.diff(distance_km) <= 0):
    raise ValueError("Distances must be strictly increasing.")

# ============================================================
# 3. CALCULATE VOLTAGE PERFORMANCE INDICATORS
# ============================================================

baseline_drop_pct = (
    (baseline[0] - baseline[-1]) / baseline[0]
) * 100

intervention_drop_pct = (
    (intervention[0] - intervention[-1])
    / intervention[0]
) * 100

receiving_end_improvement = (
    intervention[-1] - baseline[-1]
)

baseline_violations = int(
    np.sum(baseline < voltage_min)
)

intervention_violations = int(
    np.sum(intervention < voltage_min)
)

print("VOLTAGE PROFILE SUMMARY")
print("-" * 45)
print(f"Number of buses: {len(distance_km)}")
print(f"Feeder length: {distance_km[-1]:.1f} km")
print(f"Baseline receiving-end voltage: {baseline[-1]:.3f} p.u.")
print(
    "Intervention receiving-end voltage: "
    f"{intervention[-1]:.3f} p.u."
)
print(f"Baseline voltage drop: {baseline_drop_pct:.2f}%")
print(f"Intervention voltage drop: {intervention_drop_pct:.2f}%")
print(
    "Receiving-end voltage improvement: "
    f"{receiving_end_improvement:.3f} p.u."
)
print(f"Baseline buses below reference: {baseline_violations}")
print(
    "Intervention buses below reference: "
    f"{intervention_violations}"
)

# ============================================================
# 4. PLOT
# ============================================================

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(
    distance_km,
    baseline,
    linewidth=2.5,
    marker="o",
    markersize=6,
    label="Baseline: aged conductor, PF = 0.75",
    zorder=3,
)

ax.plot(
    distance_km,
    intervention,
    linewidth=2.5,
    marker="s",
    markersize=6,
    label="Intervention: upgraded conductor, capacitor, "
          "balanced load",
    zorder=3,
)

ax.axhline(
    voltage_min,
    linestyle=":",
    linewidth=2,
    label="Reference voltage = 0.95 p.u.",
    zorder=2,
)

# Identify the buses below the reference voltage.
mask_baseline = baseline < voltage_min
mask_intervention = intervention < voltage_min

ax.scatter(
    distance_km[mask_baseline],
    baseline[mask_baseline],
    marker="x",
    s=65,
    label="Baseline buses below reference",
    zorder=4,
)

ax.set_title(
    "Voltage Profile Along a Representative 11 kV Feeder",
    fontsize=13,
    fontweight="bold",
)

ax.set_xlabel("Distance from injection substation (km)", fontsize=11)
ax.set_ylabel("Bus voltage (p.u.)", fontsize=11)

ax.set_xlim(distance_km.min(), distance_km.max())
ax.set_ylim(0.75, 1.02)
ax.set_xticks(np.arange(0, 15.1, 1.5))
ax.set_yticks(np.arange(0.75, 1.01, 0.05))

ax.grid(linestyle=":", alpha=0.5)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.legend(loc="lower left", frameon=False, fontsize=9)

fig.tight_layout()

# Create output directory automatically.
output_dir = Path("data")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "voltage_profile_11kv.png"

fig.savefig(output_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"\nFigure saved to: {output_path.resolve()}")


import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ============================================================
# 1. SIMULATION RESULTS
# Replace these illustrative values with actual pandapower
# results before reporting them as validated simulation outputs.
# ============================================================

labels = ["Baseline", "Combined intervention"]
values = np.array([17.32, 2.62], dtype=float)
colors = ["#c62828", "#2e7d32"]   # red, green

# ============================================================
# 2. VALIDATE DATA AND CALCULATE IMPROVEMENT
# ============================================================

if len(labels) != len(values):
    raise ValueError("Labels and values must have equal lengths.")

if not np.all(np.isfinite(values)):
    raise ValueError("Technical-loss values must be finite.")

if np.any(values < 0):
    raise ValueError("Technical losses cannot be negative.")

if values[0] == 0:
    raise ValueError(
        "Relative reduction cannot be calculated from a zero baseline."
    )

absolute_reduction = values[0] - values[1]
relative_reduction = absolute_reduction / values[0] * 100

print("TECHNICAL-LOSS COMPARISON")
print("-" * 40)
print(f"Baseline loss: {values[0]:.2f}%")
print(f"Intervention loss: {values[1]:.2f}%")
print(f"Absolute reduction: {absolute_reduction:.2f} percentage points")
print(f"Relative reduction: {relative_reduction:.2f}%")

# ============================================================
# 3. CREATE FIGURE
# ============================================================

fig, ax = plt.subplots(figsize=(8, 5.5))

# Matplotlib's default colour cycle is used.
bars = ax.bar(
    labels,
    values,
    color=colors,
    width=0.55,
    zorder=3,
)

# Add value labels above bars.
for bar, value in zip(bars, values):
    ax.annotate(
        f"{value:.2f}%",
        xy=(bar.get_x() + bar.get_width() / 2, value),
        xytext=(0, 6),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=12,
        fontweight="bold",
    )

# Add relative improvement annotation.
ax.text(
    0.98,
    0.95,
    f"Relative reduction: {relative_reduction:.2f}%",
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=10,
)

ax.set_title(
    "Simulated Technical Loss Before and After Intervention",
    fontsize=12,
    fontweight="bold",
    pad=12,
)

ax.set_ylabel("Technical loss (% of energy delivered)", fontsize=11)

ax.set_ylim(0, max(20, values.max() * 1.15))
ax.set_yticks(np.arange(0, 21, 2.5))

ax.grid(axis="y", linestyle=":", alpha=0.5, zorder=0)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.tight_layout()

# ============================================================
# 4. SAVE HIGH-RESOLUTION OUTPUT
# ============================================================

output_dir = Path("data")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "technical_loss_comparison.png"

fig.savefig(
    output_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print(f"\nFigure saved to: {output_path.resolve()}")



import numpy as np
import pandas as pd
import sklearn

from sklearn.ensemble import IsolationForest

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("NTL detection dependencies installed successfully.")

"""
Synthetic Non-Technical Loss (NTL) Detection
Smart Grid Decision Support System (SG-DSS)

DATA HONESTY:
- All consumption readings are synthetic.
- Theft-like anomalies are injected for controlled evaluation.
- Labels identify injected anomalies, not confirmed real-world theft.
- Reported performance is synthetic-data performance only.

Model:
- Isolation Forest (unsupervised fitting).
- Stratified hold-out evaluation by meter.
- The held-out labels are not used to fit the model or select
  the prediction threshold.

Outputs:
    data/ntl_meter_scores.csv
    data/ntl_consumption_matrix.npy
    data/ntl_metrics.json
    data/ntl_pattern_metrics.csv
    data/ntl_confusion_matrix.csv
    data/ntl_feature_data.csv
"""

from pathlib import Path
import json

import numpy as np
import pandas as pd

from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    balanced_accuracy_score,
    accuracy_score,
)


# ============================================================
# 1. CONFIGURATION
# ============================================================

SEED = 42
N_METERS = 200
N_DAYS = 120
THEFT_FRACTION = 0.15
TEST_SIZE = 0.30
CONTAMINATION = THEFT_FRACTION

OUTPUT_DIR = Path("data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(SEED)


# ============================================================
# 2. GENERATE SYNTHETIC CUSTOMER CONSUMPTION
# ============================================================

customer_classes = rng.choice(
    ["residential", "small_commercial"],
    size=N_METERS,
    p=[0.80, 0.20],
)

base_kwh = np.where(
    customer_classes == "residential",
    rng.normal(6.5, 1.8, N_METERS),
    rng.normal(22.0, 6.0, N_METERS),
)

base_kwh = np.clip(base_kwh, 1.0, None)

days = np.arange(N_DAYS)

# Synthetic calendar convention:
# Days 5 and 6 of each seven-day cycle represent weekends.
weekend_mask = (days % 7 >= 5).astype(float)

data = np.zeros((N_METERS, N_DAYS), dtype=float)

for meter_idx in range(N_METERS):

    weekend_effect = (
        1.0
        + 0.10
        * weekend_mask
        * (
            1
            if customer_classes[meter_idx] == "residential"
            else -1
        )
    )

    seasonal_effect = (
        1.0 + 0.08 * np.sin(2 * np.pi * days / 30)
    )

    noise = rng.normal(1.0, 0.07, N_DAYS)

    data[meter_idx] = (
        base_kwh[meter_idx]
        * weekend_effect
        * seasonal_effect
        * noise
    )

data = np.clip(data, 0.2, None)


# ============================================================
# 3. INJECT KNOWN SYNTHETIC ANOMALIES
# ============================================================

n_theft = int(N_METERS * THEFT_FRACTION)

theft_meter_ids = rng.choice(
    N_METERS,
    size=n_theft,
    replace=False,
)

anomaly_types = [
    "step_down",
    "partial_siphon",
    "flat_line",
    "zero_weekend",
]

theft_type = {}
anomaly_start_day = {}

for meter_idx in theft_meter_ids:

    meter_idx = int(meter_idx)

    pattern = str(rng.choice(anomaly_types))
    start = int(rng.integers(30, 80))

    theft_type[meter_idx] = pattern
    anomaly_start_day[meter_idx] = start

    if pattern == "step_down":

        # Sudden, sustained reduction in recorded consumption.
        data[meter_idx, start:] *= rng.uniform(0.25, 0.45)

    elif pattern == "partial_siphon":

        # A proportion of consumption is not recorded.
        data[meter_idx, start:] *= rng.uniform(0.50, 0.70)

    elif pattern == "flat_line":

        # Suspiciously constant, low readings.
        data[meter_idx, start:] = rng.uniform(0.5, 1.2)

    elif pattern == "zero_weekend":

        # Strong consumption reduction on weekends only.
        affected_days = (
            (days >= start)
            & (days % 7 >= 5)
        )

        data[meter_idx, affected_days] *= 0.05


labels = np.zeros(N_METERS, dtype=int)
labels[theft_meter_ids] = 1

# Each meter receives its own stable identifier.
meter_ids = np.array(
    [f"MTR-{i:04d}" for i in range(N_METERS)]
)

# Use an explicit "none" category for normal meters.
pattern_labels = np.array([
    theft_type.get(i, "none")
    for i in range(N_METERS)
])

start_days = np.array([
    anomaly_start_day.get(i, -1)
    for i in range(N_METERS)
])


# ============================================================
# 4. FEATURE ENGINEERING
# ============================================================

def features_for_meter(series):
    """
    Calculate seven features from one meter's daily kWh readings.

    The features are meter-level summaries. They are not proof
    of electricity theft.
    """

    series = np.asarray(series, dtype=float)

    if series.ndim != 1 or series.size < 14:
        raise ValueError(
            "Each meter needs at least 14 daily observations."
        )

    if not np.all(np.isfinite(series)):
        raise ValueError(
            "Consumption readings contain NaN or infinity."
        )

    eps = 1e-9

    first_half = series[: series.size // 2]
    second_half = series[series.size // 2 :]

    mean_kwh = float(np.mean(series))
    std_kwh = float(np.std(series))

    coeff_variation = std_kwh / (mean_kwh + eps)

    trend_ratio = (
        (np.mean(second_half) - np.mean(first_half))
        / (np.mean(first_half) + eps)
    )

    min_to_mean_ratio = (
        np.min(series) / (mean_kwh + eps)
    )

    day_index = np.arange(series.size)

    weekday_values = series[day_index % 7 < 5]
    weekend_values = series[day_index % 7 >= 5]

    weekday_mean = float(np.mean(weekday_values))
    weekend_mean = float(np.mean(weekend_values))

    weekend_to_weekday_ratio = (
        weekend_mean / (weekday_mean + eps)
    )

    # Compare average consumption across complete seven-day weeks.
    complete_days = (series.size // 7) * 7

    weekly_means = (
        series[:complete_days]
        .reshape(-1, 7)
        .mean(axis=1)
    )

    if weekly_means.size >= 2:
        week_on_week_volatility = float(
            np.std(np.diff(weekly_means))
        )
    else:
        week_on_week_volatility = 0.0

    # Relative flatness: high values indicate low variation
    # relative to the meter's mean consumption.
    flatness = mean_kwh / (std_kwh + eps)

    features = np.array([
        mean_kwh,
        coeff_variation,
        trend_ratio,
        min_to_mean_ratio,
        weekend_to_weekday_ratio,
        week_on_week_volatility,
        flatness,
    ], dtype=float)

    if not np.all(np.isfinite(features)):
        raise ValueError(
            "Feature calculation generated a non-finite value."
        )

    return features


FEATURE_NAMES = [
    "mean_kwh",
    "coeff_variation",
    "trend_ratio",
    "min_to_mean_ratio",
    "weekend_to_weekday_ratio",
    "week_on_week_volatility",
    "flatness",
]

X = np.vstack([
    features_for_meter(data[i])
    for i in range(N_METERS)
])

feature_df = pd.DataFrame(X, columns=FEATURE_NAMES)

feature_df.insert(0, "meter_id", meter_ids)
feature_df["customer_class"] = customer_classes
feature_df["true_label"] = labels
feature_df["injected_pattern"] = pattern_labels
feature_df["anomaly_start_day"] = start_days


# ============================================================
# 5. STRATIFIED TRAIN/TEST SPLIT BY METER
# ============================================================

# Stratification preserves the normal/anomalous class proportion
# approximately in both partitions.
all_indices = np.arange(N_METERS)

train_idx, test_idx = train_test_split(
    all_indices,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=labels,
)

X_train = X[train_idx]
X_test = X[test_idx]

y_train = labels[train_idx]
y_test = labels[test_idx]

# NOTE:
# y_train and y_test are used only to describe and evaluate the
# experiment. They are never passed to IsolationForest.fit().

if np.unique(y_test).size != 2:
    raise ValueError(
        "The test partition must contain both normal and "
        "anomalous meters to calculate ROC-AUC."
    )


# ============================================================
# 6. FIT ISOLATION FOREST
# ============================================================

model = IsolationForest(
    n_estimators=300,
    contamination=CONTAMINATION,
    random_state=SEED,
    n_jobs=-1,
)

# Unsupervised training: no ground-truth labels supplied.
model.fit(X_train)

# score_samples: lower values mean more anomalous.
# Negation makes higher scores mean more anomalous.
train_scores = -model.score_samples(X_train)
test_scores = -model.score_samples(X_test)

# The decision threshold is learned from the training partition.
# Test labels do not determine the threshold.
train_predictions = (model.predict(X_train) == -1).astype(int)
test_predictions = (model.predict(X_test) == -1).astype(int)

# Score and flag all meters for the operational ranking file.
all_scores = -model.score_samples(X)
all_predictions = (model.predict(X) == -1).astype(int)

split_label = np.full(N_METERS, "train", dtype=object)
split_label[test_idx] = "test"


# ============================================================
# 7. ROBUST EVALUATION METRICS
# ============================================================

def calculate_metrics(y_true, y_pred, scores):
    """
    Calculate binary-classification metrics safely.

    Labels:
        0 = no injected anomaly
        1 = injected anomaly

    Higher anomaly scores indicate greater suspicion.
    """

    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    scores = np.asarray(scores, dtype=float)

    if not (
        len(y_true) == len(y_pred) == len(scores)
    ):
        raise ValueError(
            "Labels, predictions, and scores must have equal lengths."
        )

    if not np.all(np.isfinite(scores)):
        raise ValueError("Anomaly scores contain non-finite values.")

    # Explicit label order guarantees a stable 2x2 matrix.
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    ).ravel()

    precision = precision_score(
        y_true, y_pred, zero_division=0
    )

    recall = recall_score(
        y_true, y_pred, zero_division=0
    )

    f1 = f1_score(
        y_true, y_pred, zero_division=0
    )

    specificity = (
        float(tn / (tn + fp))
        if (tn + fp) > 0 else 0.0
    )

    negative_predictive_value = (
        float(tn / (tn + fn))
        if (tn + fn) > 0 else 0.0
    )

    # ROC-AUC and average precision require both classes.
    if np.unique(y_true).size == 2:
        roc_auc = float(
            roc_auc_score(y_true, scores)
        )

        avg_precision = float(
            average_precision_score(y_true, scores)
        )
    else:
        roc_auc = None
        avg_precision = None

    return {
        "n_evaluated": int(len(y_true)),
        "n_normal": int(np.sum(y_true == 0)),
        "n_anomalous": int(np.sum(y_true == 1)),
        "precision": float(precision),
        "recall": float(recall),
        "f1_score": float(f1),
        "roc_auc": roc_auc,
        "average_precision": avg_precision,
        "accuracy": float(
            accuracy_score(y_true, y_pred)
        ),
        "specificity": specificity,
        "negative_predictive_value": negative_predictive_value,
        "balanced_accuracy": float(
            balanced_accuracy_score(y_true, y_pred)
        ),
        "true_positives": int(tp),
        "false_positives": int(fp),
        "true_negatives": int(tn),
        "false_negatives": int(fn),
    }


test_metrics = calculate_metrics(
    y_test,
    test_predictions,
    test_scores,
)

train_metrics = calculate_metrics(
    y_train,
    train_predictions,
    train_scores,
)


# ============================================================
# 8. ANOMALY-PATTERN BREAKDOWN ON HELD-OUT METERS
# ============================================================

pattern_rows = []

for pattern in anomaly_types:

    # Only test-set meters with this injected pattern.
    pattern_indices = test_idx[
        pattern_labels[test_idx] == pattern
    ]

    n_pattern = len(pattern_indices)

    if n_pattern == 0:
        pattern_rows.append({
            "pattern": pattern,
            "n_test_meters": 0,
            "caught": 0,
            "missed": 0,
            "recall": None,
            "status": "No examples in test partition",
        })
        continue

    caught = int(
        np.sum(all_predictions[pattern_indices] == 1)
    )

    missed = int(n_pattern - caught)

    pattern_rows.append({
        "pattern": pattern,
        "n_test_meters": int(n_pattern),
        "caught": caught,
        "missed": missed,
        "recall": float(caught / n_pattern),
        "status": "Evaluated",
    })

pattern_df = pd.DataFrame(pattern_rows)


# ============================================================
# 9. CONFUSION MATRIX TABLE
# ============================================================

tn, fp, fn, tp = confusion_matrix(
    y_test,
    test_predictions,
    labels=[0, 1],
).ravel()

confusion_df = pd.DataFrame(
    [
        [int(tn), int(fp)],
        [int(fn), int(tp)],
    ],
    index=["Actual normal", "Actual anomalous"],
    columns=["Predicted normal", "Predicted anomalous"],
)

confusion_df.index.name = "Actual / predicted"


# ============================================================
# 10. EXPORT RESULTS
# ============================================================

# Full meter-level ranking. Predictions for the training partition
# are descriptive; headline performance is calculated on test only.
scores_df = feature_df.copy()

scores_df["partition"] = split_label
scores_df["anomaly_score"] = all_scores
scores_df["predicted_flag"] = all_predictions

scores_df["evaluation_role"] = np.where(
    split_label == "test",
    "Held-out evaluation",
    "Training / exploratory",
)

scores_df = scores_df.sort_values(
    "anomaly_score",
    ascending=False,
)

scores_path = OUTPUT_DIR / "ntl_meter_scores.csv"
scores_df.to_csv(scores_path, index=False)

# Save the synthetic consumption matrix (meters x days).
matrix_path = OUTPUT_DIR / "ntl_consumption_matrix.npy"
np.save(matrix_path, data)

# Export engineered features and metadata.
features_path = OUTPUT_DIR / "ntl_feature_data.csv"
feature_df.to_csv(features_path, index=False)

# Export pattern-specific held-out results.
pattern_path = OUTPUT_DIR / "ntl_pattern_metrics.csv"
pattern_df.to_csv(pattern_path, index=False)

# Export confusion matrix.
confusion_path = OUTPUT_DIR / "ntl_confusion_matrix.csv"
confusion_df.to_csv(confusion_path)

# Record the experimental design and results.
metrics = {
    "data_honesty_note": (
        "Synthetic data only. Injected anomalies are not "
        "confirmed real-world electricity theft."
    ),
    "random_seed": SEED,
    "n_meters": N_METERS,
    "n_days": N_DAYS,
    "n_daily_observations": int(N_METERS * N_DAYS),
    "n_injected_anomalies": int(n_theft),
    "n_normal_meters": int(N_METERS - n_theft),
    "test_size": TEST_SIZE,
    "n_train_meters": int(len(train_idx)),
    "n_test_meters": int(len(test_idx)),
    "model": "IsolationForest",
    "n_estimators": 300,
    "contamination": CONTAMINATION,
    "feature_names": FEATURE_NAMES,
    "evaluation_method": (
        "Stratified meter-level hold-out. Isolation Forest "
        "fitted on training features only. Training-derived "
        "threshold applied to the held-out test partition. "
        "Synthetic labels used for evaluation only."
    ),
    "training_metrics_descriptive_only": train_metrics,
    "held_out_test_metrics": test_metrics,
    "test_pattern_breakdown": pattern_rows,
    "injected_pattern_counts_all_meters": {
        pattern: int(sum(
            theft_type.get(int(i)) == pattern
            for i in theft_meter_ids
        ))
        for pattern in anomaly_types
    },
    "interpretation_limitations": [
        "Synthetic performance is not real-world validation.",
        "Injected anomalies may be easier to detect than real theft.",
        "Contamination is configured using the known synthetic "
        "anomaly fraction; this is an experimental assumption.",
        "Small held-out sample sizes make metrics uncertain.",
        "A flagged meter is a candidate for investigation, not "
        "proof of electricity theft.",
    ],
}

metrics_path = OUTPUT_DIR / "ntl_metrics.json"

with open(metrics_path, "w", encoding="utf-8") as file:
    json.dump(metrics, file, indent=2, allow_nan=False)


# ============================================================
# 11. DISPLAY SUMMARY
# ============================================================

print("=" * 65)
print("SYNTHETIC NTL DETECTION — EXPERIMENT SUMMARY")
print("=" * 65)

print(f"Total meters:          {N_METERS}")
print(f"Observation days:      {N_DAYS}")
print(f"Injected anomalies:    {n_theft}")
print(f"Training meters:       {len(train_idx)}")
print(f"Held-out test meters:  {len(test_idx)}")

print("\nHELD-OUT TEST METRICS")
print("-" * 65)

for key, value in test_metrics.items():
    if isinstance(value, float):
        display_value = (
            f"{value:.4f}"
            if np.isfinite(value)
            else "Not available"
        )
    else:
        display_value = str(value)

    print(f"{key:28s}: {display_value}")

print("\nTEST CONFUSION MATRIX")
print(confusion_df.to_string())

print("\nHELD-OUT RECALL BY INJECTED PATTERN")
print(pattern_df.to_string(index=False))

print("\nTOP 10 FLAGGED METERS — RANKED BY ANOMALY SCORE")

print(
    scores_df[
        [
            "meter_id",
            "customer_class",
            "injected_pattern",
            "partition",
            "true_label",
            "predicted_flag",
            "anomaly_score",
        ]
    ]
    .head(10)
    .to_string(index=False)
)

print("\nOUTPUT FILES")
for path in [
    scores_path,
    matrix_path,
    features_path,
    pattern_path,
    confusion_path,
    metrics_path,
]:
    print(f"- {path.resolve()}")

print("\nExperiment completed successfully.")


